## **03_self_attention: Sell Attention**
We are still having the `River Bank` vs `Money Bank` problem  
No way yet to adjust a word's meaning based on its context 
This is the heart of the transformer  

For this, let's focus on this example of the ambiguous word `Crane`  
1. The crane ate a fish - crane = a bird  
2. The crane lifted the steel - crane = a machine  

Right now the starting vector for `Crane` is the same for both  
We need a mechanism to update this vector based on its neighbors  
<center><h1>Self Attention</h1></center>

This is the formula:  
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

The whole process is 3 steps:
1. Scoring
2. Normalizing
3. Aggregating

### First, what are Q, K and V?  
They are three distinct vectors for every single word created by projecting the input vector `x`

+ `Query (Q)` the word's search query. It's what it's looking for. 
+ `Key (K)` the word's label or keyword. It's what it is. 
+ `Value (V)` the word's payload, the information it offers.  

Now, you might be asking a brilliant question. Why do we need a separate value vector?   
Isn't our input vector X already the information?   
The V vector is a transformed version of X packaged for others to consume. Look at this table. The input vector X is like your entire knowledge and resume. It's the raw complete information. But the value vector V, that's your prepared elevator pitch. It's packaged consumable information that's ready to be aggregated. It's made from a learned transformation of X.   

The model learns the best elevator pitch for each word. It is like without V, how would X be modified right.  

You will see more clearly, in the next section - multi-headed attention


Okay, analogy over. Let's see the numbers.  
<center>We'll use a dead simple 2D space:</center>

+ Dimension 1: represents `is it an animal?`
+ dimension 2: represents `is it a machine?`. 

Here are the vectors for our words. Check out this table. Notice the crane vectors are all ambiguous like 7.7. But ate and fish are high on the animal

| Token  | Q - "I'm looking for..." | K - "I am..."          | V - "I offer this info..."  |
|--------|--------------------------|------------------------|-----------------------------|
| crane  | [0.7, 0.7]               | [0.7, 0.7]             | [0.5, 0.5] (Ambiguous)      |
| ate    | ...                      | [0.9, 0.1] (High Animal) | [0.9, 0.1]               |
| fish   | ...                      | [0.8, 0.2] (High Animal) | [0.8, 0.2]               |
| lifted | ...                      | [0.1, 0.9] (High Machine) | [0.1, 0.9]              |
| steel  | ...                      | [0.2, 0.8] (High Machine) | [0.2, 0.8]              |

<center><h3>Sentence 1: "The crane ate a fish"</h3></center>

+ crane Query: [0.7, 0.7]  
+ crane Key: [0.7, 0.7] | ate Key: [0.9, 0.1] | fish Key: [0.8, 0.2]   

1. Scoring (QK<sup>T</sup>): "Crane" probes every word's Key

Score(crane → crane): [0.7, 0.7] · [0.7, 0.7] = **0.98** (High self-affinity)

Score(crane → ate): [0.7, 0.7] · [0.9, 0.1] = **0.70** (High match!)

Score(crane → fish): [0.7, 0.7] · [0.8, 0.2] = **0.70** (High match!)

2. Normalizing(`softmax`): Convert scores to percentages
<center>Raw score: <strong>[0.98, 0.7, 0.7]</strong></center>
<center>&darr;</center>
<center>Attention weights: <strong>[0.4, 0.3, 0.3]</strong></center>

#### Yeah, so now "Crane" will listen:

+ 40% to its original self
+ 30% to `ate`
+ 30% to `fish`

3. Aggregating (`...V`): weighted sum of value vectors
```
New_Vector(crane) =
  0.4 * [0.5, 0.5] (from crane) +
  0.3 * [0.9, 0.1] (from ate) +
  0.3 * [0.8, 0.2] (from fish)

New_Vector(crane) = [0.20, 0.20] + [0.27, 0.03] + [0.24, 0.06]
                  = [0.71, 0.29]
```
Now skewed heavily towards animal dimension 1

<center><h3>Sentence 2: "The crane lifted the steel"</h3></center>

+ crane Query: [0.7, 0.7]  
+ crane Key: [0.7, 0.7] | lifted Key: [0.1, 0.9] | steel Key: [0.2, 0.8]   

Score(crane → crane): [0.7, 0.7] · [0.7, 0.7] = **0.98**

Score(crane → lifted): [0.7, 0.7] · [0.1, 0.9] = **0.70** (High match!)

Score(crane → steel): [0.7, 0.7] · [0.2, 0.8] = **0.70** (High match!)

2. Normalizing(`softmax`): Convert scores to percentages
<center>Raw score: <strong>[0.98, 0.7, 0.7]</strong></center>
<center>&darr;</center>
<center>Attention weights: <strong>[0.4, 0.3, 0.3]</strong></center>

3. Aggregating (`...V`): same weights, different values
```
New_Vector(crane) =
  0.4 * [0.5, 0.5] (from crane) +
  0.3 * [0.1, 0.9] (from lifted) +
  0.3 * [0.2, 0.8] (from steel)

New_Vector(crane) = [0.20, 0.20] + [0.03, 0.27] + [0.06, 0.24]
                  = [0.29, 0.71]
```
Now instead, skewed heavily towards machine dimension 2

The exact same initial crane vector has been transformed into two completely different contextaware vectors. This is the power of `self attention`. It's a communication mechanism that allows every word to dynamically pull in information from its context to refine its own meaning. 

We've built the intuition. We saw how the word `crane` could resolve its ambiguity by listening to its neighbors. Now it's time to translate that exact process into the language of linear algebra. 

Time to build, step by step:  
$$ \text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass

B, T, C = 1, 4, 2  # Batch, Time (sequence length), Channels (embedding dim)
x = torch.tensor([
    [[0.1, 0.1],   # A
     [1.0, 0.2],   # crane (mostly object, slightly action)
     [0.1, 0.9],   # ate (mostly action)
     [0.8, 0.0]]   # fish (purely object)
]).float()

#### **Step 1: Projecting `x` into Q, K, and V**
To get our Query, Key, and Value vectors, we use **learnable** linear transformations. These `nn.Linear` layers are the "brains" of the operation; their weights are updated during training. 

In [9]:
# The learnable components
q_proj = nn.Linear(C, C, bias=False)
k_proj = nn.Linear(C, C, bias=False)
v_proj = nn.Linear(C, C, bias=False)

# Manually set weights for this tutorial
torch.manual_seed(42)
q_proj.weight.data = torch.randn(C, C)
k_proj.weight.data = torch.randn(C, C)
v_proj.weight.data = torch.randn(C, C)

# --- Perform the projections ---
q = q_proj(x)
k = k_proj(x)
v = v_proj(x)
print("x:", x.shape)
print("q:", q.shape)

x: torch.Size([1, 4, 2])
q: torch.Size([1, 4, 2])


Let's track our tensor shapes and their meaning.

| Variable | Shape `(B, T, C)` | Meaning |
| :--- | :--- | :--- |
| `x` | `(1, 4, 2)` | The batch of raw input vectors. |
| `q` | `(1, 4, 2)` | The "Query" vector for each of the 4 tokens. |
| `k` | `(1, 4, 2)` | The "Key" vector for each of the 4 tokens. |
| `v` | `(1, 4, 2)` | The "Value" vector for each of the 4 tokens. |

#### **Step 2: Calculate Attention Scores (`q @ k.transpose`)**
This is the core of the communication. We need to compute the dot product of every token's query with every other token's key. 

*   `q` shape: `(1, 4, 2)`.
*   `k` has shape `(1, 4, 2)`.
*   `k.transpose(-2, -1)` results in a shape of `(1, 2, 4)`.
*   The multiplication is `(1, 4, 2) @ (1, 2, 4)`, which results in a `(1, 4, 4)` matrix.

In [11]:
# --- Score Calculation ---
scores = q @ k.transpose(-2, -1)

print("--- Raw Scores (Attention Matrix) for (A, crane, ate, fish) ---")
print(scores.shape)
print(scores)

--- Raw Scores (Attention Matrix) for (A, crane, ate, fish) ---
torch.Size([1, 4, 4])
tensor([[[ 0.0012,  0.0427, -0.0295,  0.0403],
         [-0.0034,  0.1632, -0.2006,  0.1700],
         [ 0.0166,  0.3065, -0.1234,  0.2732],
         [-0.0058,  0.0778, -0.1417,  0.0894]]], grad_fn=<UnsafeViewBackward0>)


This `(4, 4)` matrix holds the raw compatibility scores. For example, the query for "crane" (row 1) has the highest compatibility with the key for "crane" (column 1)

#### **Step 3 & 4: Scale and Softmax**
We scale the scores for stability, then use `softmax` to turn them into attention weights that sum to 1 for each row.

In [15]:
d_k = k.size(-1)
print("d_k:", d_k)
scaled_scores = scores / math.sqrt(d_k)
attention_weights = F.softmax(scaled_scores, dim=-1) # Softmax along the rows
print("attention_weights: ", attention_weights)

d_k: 2
attention_weights:  tensor([[[0.2477, 0.2551, 0.2424, 0.2547],
         [0.2424, 0.2727, 0.2109, 0.2740],
         [0.2308, 0.2833, 0.2091, 0.2768],
         [0.2476, 0.2627, 0.2249, 0.2648]]], grad_fn=<SoftmaxBackward0>)


#### **Step 5: Aggregate the Values (`attention_weights @ v`)**
Now we use our weights to create a weighted average of the `Value` vectors.
*   `attention_weights` has shape `(1, 4, 4)`.
*   `v` has shape `(1, 4, 2)`.
*   The multiplication `(1, 4, 4) @ (1, 4, 2)` produces a final tensor of shape `(1, 4, 2)`.

In [14]:
output = attention_weights @ v

print("\n--- Final Output (Context-Aware Vectors) ---")
print(output.shape)
print(output)


--- Final Output (Context-Aware Vectors) ---
torch.Size([1, 4, 2])
tensor([[[0.3131, 0.5096],
         [0.3198, 0.5047],
         [0.3250, 0.5104],
         [0.3157, 0.5055]]], grad_fn=<UnsafeViewBackward0>)


Here is a summary of the tensor transformations:

| Step | Operation | Input Shapes | Output Shape `(B, T, ...)` | Meaning |
| :--- | :--- | :--- | :--- | :--- |
| 1 | `q_proj(x)` | `(1, 4, 2)` | `(1, 4, 2)` | Create Q, K, V for each token |
| 2 | `q @ k.T` | `(1, 4, 2)` & `(1, 2, 4)` | `(1, 4, 4)` | Raw compatibility scores |
| 3 | `/ sqrt(d_k)` | `(1, 4, 4)` | `(1, 4, 4)` | Stabilized scores |
| 4 | `softmax` | `(1, 4, 4)` | `(1, 4, 4)` | Attention probabilities |
| 5 | `att @ v` | `(1, 4, 4)` & `(1, 4, 2)` | `(1, 4, 2)` | Context-aware output vectors|

Success! We have taken our raw input `x` and produced a new tensor `output` of the exact same shape, where each token's vector has been updated with information from its neighbors.